# Layer 2 — Feature Engineering for Guwahati Urban Prediction

This notebook builds on Layer 1 composites to produce:
1. **LULC classification** — CART classifier, 4 classes (Urban, Vegetation, Water, Barren)
2. **Change detection** — post-classification comparison between epochs
3. **Proximity features** — distance to roads, water, infrastructure from OSM
4. **Feature stacks** — combined rasters ready for Layer 3 forecasting

All processing runs in GEE. Outputs export to Google Drive → `guwahati_urban_prediction/` folder.

## 0. Setup & Authentication

In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='vivid-brand-182819')
print('Earth Engine initialized:', ee.String('connected').getInfo())

BBOX = {'west': 91.55, 'south': 26.10, 'east': 91.88, 'north': 26.25}
aoi = ee.Geometry.Rectangle([BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north']])

LULC_CLASSES = {0: 'Urban', 1: 'Vegetation', 2: 'Water', 3: 'Barren'}
LULC_PALETTE = ['#e41a1c', '#4daf4a', '#377eb8', '#c2a06a']
DRIVE_FOLDER = 'guwahati_urban_prediction'

print(f'AOI area: {aoi.area().divide(1e6).getInfo():.1f} km²')
print(f'Classes: {LULC_CLASSES}')

## 1. Rebuild composites in GEE (same as Layer 1)

In [ ]:
# --- Sentinel-2 composite builder (same logic as Layer 1) ---

def cloud_mask_s2(image):
    scl = image.select('SCL')
    clear = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(clear)

def add_indices(image, nir='B8', red='B4', swir='B11', green='B3'):
    ndvi = image.normalizedDifference([nir, red]).rename('NDVI')
    ndbi = image.normalizedDifference([swir, nir]).rename('NDBI')
    ndwi = image.normalizedDifference([green, nir]).rename('NDWI')
    return image.addBands([ndvi, ndbi, ndwi])

S2_BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDBI', 'NDWI']

def build_s2_composite(start, end):
    col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
        .map(cloud_mask_s2)
        .map(lambda img: add_indices(img))
    )
    return col.select(S2_BANDS).median().clip(aoi).toFloat()

# --- Landsat composite builder ---

def cloud_mask_landsat(image):
    qa = image.select('QA_PIXEL')
    return image.updateMask(
        qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    )

def scale_l8(image):
    optical = image.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']).multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True)

def scale_l5(image):
    optical = image.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']).multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True)

def build_landsat_composite(start, end, sensor='L8'):
    if sensor == 'L8':
        collection = 'LANDSAT/LC08/C02/T1_L2'
        sfn = scale_l8
        bands = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']
        nir, red, swir, green = 'SR_B5', 'SR_B4', 'SR_B6', 'SR_B3'
    else:
        collection = 'LANDSAT/LT05/C02/T1_L2'
        sfn = scale_l5
        bands = ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']
        nir, red, swir, green = 'SR_B4', 'SR_B3', 'SR_B5', 'SR_B2'

    _nir, _red, _swir, _green = nir, red, swir, green
    def _add_idx(image, _nir=_nir, _red=_red, _swir=_swir, _green=_green):
        return add_indices(image, _nir, _red, _swir, _green)

    col = (
        ee.ImageCollection(collection)
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUD_COVER', 30))
        .map(cloud_mask_landsat)
        .map(sfn)
        .map(_add_idx)
    )
    out_bands = bands + ['NDVI', 'NDBI', 'NDWI']
    return col.select(out_bands).median().clip(aoi).toFloat()

# Build all composites — Sentinel-2 for recent, Landsat for historical
# We use Landsat for classification since it covers all epochs at consistent 30m

EPOCHS = {
    '2000': ('2000-11-01', '2001-04-30', 'L5'),
    '2005': ('2005-11-01', '2006-04-30', 'L5'),
    '2010': ('2010-11-01', '2011-04-30', 'L5'),
    '2016': ('2016-11-01', '2017-04-30', 'L8'),
    '2021': ('2021-11-01', '2022-04-30', 'L8'),
}

composites = {}
for label, (start, end, sensor) in EPOCHS.items():
    comp = build_landsat_composite(start, end, sensor)
    band_count = comp.bandNames().size().getInfo()
    composites[label] = comp
    print(f'[Composite] {label} ({sensor}): {band_count} bands ready')

print(f'\n{len(composites)} composites built in GEE (server-side, no download needed)')

## 2. Generate training samples from spectral thresholds

In [ ]:
# Generate pseudo-training samples using spectral thresholds + known locations
# The 2021 Landsat-8 composite is used as the reference for training

ref = composites['2021']
ndvi = ref.select('NDVI')
ndbi = ref.select('NDBI')
ndwi = ref.select('NDWI')

# --- Threshold-based pseudo-labels ---
# Urban: high NDBI, low NDVI
urban_mask = ndbi.gt(0.0).And(ndvi.lt(0.25))
# Vegetation: high NDVI
veg_mask = ndvi.gt(0.35).And(ndbi.lt(0.0))
# Water: high NDWI (Brahmaputra, Deepor Beel, wetlands)
water_mask = ndwi.gt(0.1).And(ndvi.lt(0.15))
# Barren: low in all indices (sandbars, fallow land)
barren_mask = ndvi.lt(0.15).And(ndbi.lt(0.05)).And(ndwi.lt(0.0))

# Create a single label image: 0=Urban, 1=Vegetation, 2=Water, 3=Barren
label_image = (
    ee.Image(0).where(urban_mask, 0)
    .where(veg_mask, 1)
    .where(water_mask, 2)
    .where(barren_mask, 3)
    .rename('label')
    .toUint8()
)

# Mask out ambiguous pixels (where no threshold fired cleanly)
any_class = urban_mask.Or(veg_mask).Or(water_mask).Or(barren_mask)
label_image = label_image.updateMask(any_class)

# Sample training points — stratified, 500 per class
training_bands = ref.bandNames()
training_input = ref.addBands(label_image)

training_samples = training_input.stratifiedSample(
    numPoints=500,
    classBand='label',
    region=aoi,
    scale=30,
    seed=42,
    geometries=True,
)

sample_count = training_samples.size().getInfo()
print(f'Total training samples: {sample_count}')

# Check class distribution
for cls_id, cls_name in LULC_CLASSES.items():
    n = training_samples.filter(ee.Filter.eq('label', cls_id)).size().getInfo()
    print(f'  Class {cls_id} ({cls_name}): {n} samples')

## 3. Train CART classifier & classify all epochs

In [ ]:
# Train a CART classifier on the 2021 reference, then apply to all epochs
# CART is fast, interpretable, and works well for 4-class LULC in GEE

# Use only the spectral + index bands for classification (not QA bands)
classify_bands = ['NDVI', 'NDBI', 'NDWI']

classifier = ee.Classifier.smileCart(maxNodes=50).train(
    features=training_samples,
    classProperty='label',
    inputProperties=classify_bands,
)

print('CART classifier trained on 2021 reference composite')
print(f'Input features: {classify_bands}')

# Classify all epochs
lulc_maps = {}
lulc_tasks = []

for label, comp in composites.items():
    classified = comp.select(classify_bands).classify(classifier).toUint8().rename('lulc')
    lulc_maps[label] = classified

    task = ee.batch.Export.image.toDrive(
        image=classified,
        description=f'lulc_{label}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'lulc_{label}',
        region=aoi,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e9,
        fileFormat='GeoTIFF',
    )
    task.start()
    lulc_tasks.append(task)
    print(f'[LULC] Export started: lulc_{label}')

print(f'\n{len(lulc_tasks)} LULC classification tasks submitted')

## 4. Accuracy assessment

In [ ]:
# Accuracy assessment — hold out 30% of training samples for validation

split = training_samples.randomColumn('random', 42)
train_set = split.filter(ee.Filter.lt('random', 0.7))
test_set = split.filter(ee.Filter.gte('random', 0.7))

val_classifier = ee.Classifier.smileCart(maxNodes=50).train(
    features=train_set,
    classProperty='label',
    inputProperties=classify_bands,
)

validated = test_set.classify(val_classifier)
conf_matrix = validated.errorMatrix('label', 'classification')

print('=== Confusion Matrix ===')
print(conf_matrix.getInfo())
print(f'\nOverall accuracy: {conf_matrix.accuracy().getInfo():.4f}')
print(f'Kappa coefficient: {conf_matrix.kappa().getInfo():.4f}')

# Per-class accuracy
producers = conf_matrix.producersAccuracy().getInfo()
consumers = conf_matrix.consumersAccuracy().getInfo()
print('\nPer-class accuracy:')
for cls_id, cls_name in LULC_CLASSES.items():
    pa = producers[cls_id][0] if cls_id < len(producers) else 'N/A'
    ca = consumers[cls_id][0] if cls_id < len(consumers) else 'N/A'
    print(f'  {cls_name}: Producer={pa:.3f}, Consumer={ca:.3f}' if isinstance(pa, float) else f'  {cls_name}: {pa}')

## 5. Change detection between epochs

In [ ]:
# Change detection: pairwise comparison between consecutive epochs
# Output: change maps showing class transitions (from_class * 10 + to_class)
# e.g., 01 = Urban->Vegetation, 10 = Vegetation->Urban

epoch_pairs = [
    ('2000', '2005'),
    ('2005', '2010'),
    ('2010', '2016'),
    ('2016', '2021'),
]

change_tasks = []
change_maps = {}

for e1, e2 in epoch_pairs:
    lulc1 = lulc_maps[e1]
    lulc2 = lulc_maps[e2]

    # Encode transition: from_class * 10 + to_class
    change = lulc1.multiply(10).add(lulc2).rename('change').toUint8()

    # Urban gain mask (any class -> Urban)
    urban_gain = lulc1.neq(0).And(lulc2.eq(0)).rename('urban_gain').toUint8()

    # Urban loss mask (Urban -> any other class)
    urban_loss = lulc1.eq(0).And(lulc2.neq(0)).rename('urban_loss').toUint8()

    change_stack = change.addBands(urban_gain).addBands(urban_loss)
    change_maps[f'{e1}_{e2}'] = change_stack

    task = ee.batch.Export.image.toDrive(
        image=change_stack,
        description=f'change_{e1}_{e2}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'change_{e1}_{e2}',
        region=aoi,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e9,
        fileFormat='GeoTIFF',
    )
    task.start()
    change_tasks.append(task)
    print(f'[Change] Export started: change_{e1}_{e2}')

# Compute urban area stats for each epoch
print('\n=== Urban area by epoch ===')
pixel_area = ee.Image.pixelArea()
for label, lulc in lulc_maps.items():
    urban_area = lulc.eq(0).multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=30,
        maxPixels=1e9,
    ).get('lulc')
    area_km2 = ee.Number(urban_area).divide(1e6).getInfo()
    print(f'  {label}: {area_km2:.2f} km² urban')

print(f'\n{len(change_tasks)} change detection tasks submitted')

## 6. Proximity features from OSM

In [ ]:
# Proximity features — Euclidean distance to key land cover classes + terrain

# --- Distance to water (from our LULC classification) ---
water_2021 = lulc_maps['2021'].eq(2)  # class 2 = Water
dist_water = water_2021.fastDistanceTransform(256).sqrt().multiply(30).rename('dist_water_m')

# --- Distance to urban edge (from LULC) ---
urban_2021 = lulc_maps['2021'].eq(0)  # class 0 = Urban
urban_edge = urban_2021.And(
    urban_2021.focal_min(1).Not()  # pixels that are urban but neighbor non-urban
)
dist_urban_edge = urban_edge.fastDistanceTransform(256).sqrt().multiply(30).rename('dist_urban_edge_m')

# --- Road/infrastructure proximity via VIIRS night-lights ---
# VIIRS radiance correlates strongly with road density and built infrastructure
# We threshold to identify lit areas, then compute distance to them
viirs_2021 = (
    ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
    .filterBounds(aoi)
    .filterDate('2021-01-01', '2021-12-31')
    .select('avg_rad')
    .median()
    .clip(aoi)
)
# Lit pixels (radiance > 5 nW/cm²/sr) mark roads and infrastructure
lit_mask = viirs_2021.gt(5).unmask(0)
dist_roads = lit_mask.fastDistanceTransform(256).sqrt().multiply(30).rename('dist_road_m')

# --- Slope from SRTM DEM ---
srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
slope = ee.Terrain.slope(srtm).rename('slope_deg')
elevation = srtm.select('elevation')

# Stack all proximity/terrain features
proximity_stack = (
    dist_water
    .addBands(dist_urban_edge)
    .addBands(dist_roads)
    .addBands(slope)
    .addBands(elevation)
    .toFloat()
)

print('Proximity feature stack bands:')
for b in proximity_stack.bandNames().getInfo():
    print(f'  - {b}')

# Export proximity features
prox_task = ee.batch.Export.image.toDrive(
    image=proximity_stack,
    description='proximity_features_2021',
    folder=DRIVE_FOLDER,
    fileNamePrefix='proximity_features_2021',
    region=aoi,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e9,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True},
)
prox_task.start()
print('\n[Proximity] Export started: proximity_features_2021')

## 7. Export feature stacks for Layer 3

In [ ]:
# Build combined feature stacks for each epoch — input for Layer 3 forecasting
# Each stack: LULC class + spectral indices + VIIRS night-lights + proximity

viirs_years = {'2000': 2016, '2005': 2016, '2010': 2016, '2016': 2016, '2021': 2021}

feature_tasks = []
for label, comp in composites.items():
    lulc = lulc_maps[label].rename('lulc')
    indices = comp.select(['NDVI', 'NDBI', 'NDWI'])

    # VIIRS (use nearest available year)
    vy = viirs_years[label]
    viirs = (
        ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
        .filterBounds(aoi)
        .filterDate(f'{vy}-01-01', f'{vy}-12-31')
        .select('avg_rad')
        .median()
        .clip(aoi)
        .rename('nightlights')
    )

    # WorldPop (nearest year)
    wp_year = min([2000, 2010, 2015, 2020], key=lambda y: abs(y - int(label)))
    worldpop = (
        ee.ImageCollection('WorldPop/GP/100m/pop')
        .filterBounds(aoi)
        .filter(ee.Filter.eq('year', wp_year))
        .select('population')
        .mosaic()
        .clip(aoi)
        .rename('population')
    )

    # Combine everything
    feature_stack = (
        lulc.toFloat()
        .addBands(indices)
        .addBands(viirs)
        .addBands(worldpop)
        .addBands(proximity_stack)
    )

    band_names = feature_stack.bandNames().getInfo()
    print(f'[Stack] {label}: {len(band_names)} bands — {band_names}')

    task = ee.batch.Export.image.toDrive(
        image=feature_stack,
        description=f'features_{label}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'features_{label}',
        region=aoi,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e9,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
    )
    task.start()
    feature_tasks.append(task)
    print(f'[Stack] Export started: features_{label}')

print(f'\n{len(feature_tasks)} feature stack tasks submitted')

## 8. Check export task status

In [ ]:
# Check status of all Layer 2 export tasks

all_l2_tasks = lulc_tasks + change_tasks + [prox_task] + feature_tasks
print(f'Total Layer 2 export tasks: {len(all_l2_tasks)}')
print('Monitor at: https://code.earthengine.google.com/tasks\n')

completed = 0
for t in all_l2_tasks:
    status = t.status()
    state = status['state']
    desc = status['description']
    err = status.get('error_message', '')
    line = f'  {state:12s} {desc}'
    if err:
        line += f'  ERROR: {err}'
    print(line)
    if state == 'COMPLETED':
        completed += 1

print(f'\n{completed}/{len(all_l2_tasks)} completed')
if completed < len(all_l2_tasks):
    print('Re-run this cell to refresh status.')